# 加载全A和指数合并的数据

In [ ]:
# 1. 导入必要的库
import qlib
from qlib.constant import REG_CN
import logging
import os
import pandas as pd
import numpy as np
from qlib.data import D
import warnings
from pathlib import Path
warnings.filterwarnings('ignore', category=RuntimeWarning, message='Mean of empty slice')




In [ ]:
#设置参数
data_category='allAShare_with_IndexValue'
provider_url = os.path.abspath(f"./.qlib/qlib_data/{data_category}/")
qlib.init(provider_uri=provider_url, region=REG_CN, logging_level=logging.INFO)

#设置因子路径
factor_csv_file_path = '.qlib/indicator_data/allAshare_std_3m.csv'

# 回测时间范围确定
start_time = '2010-01-05'
end_time = '2024-12-31'


In [ ]:
# 指数名称到代码的映射字典
INDEX_NAME_TO_CODE = {
    'shangzheng50': '000016.SH',      # 上证50
    'hushen300': '000300.SH',         # 沪深300
    'zhongzheng500': '000905.SH',     # 中证500
    'zhongzheng1000': '000852.SH',    # 中证1000
    'allAshare':'000001.SH',
}

def extract_index_code_from_filename(file_path):
    """
    从文件路径中提取指数名称并返回对应的指数代码
    
    Args:
        file_path (str): 包含指数名称的文件路径，如 '.qlib/indicator_data/hushen300_PB.csv'
    
    Returns:
        str: 对应的指数代码，如 '000852.SH'，如果未找到则返回 None
    """
    import re
    
    # 提取文件名（不包含路径和扩展名）
    filename = os.path.basename(file_path)
    filename_without_ext = os.path.splitext(filename)[0]
    
    # 使用正则表达式提取指数名称（在第一个下划线之前的部分）
    match = re.match(r'^([a-zA-Z]+(?:[0-9]+)?)', filename_without_ext)
    if match:
        index_name = match.group(1)
        return INDEX_NAME_TO_CODE.get(index_name)
    
    return None

# 通过csv创建信号值

In [ ]:
def extract_content(s):
    """
    从路径字符串中提取最后一个/后面、第一个_后面至.csv之前的内容。
    
    参数:
        s (str): 输入的路径字符串。
        
    返回:
        str: 提取出的内容。如果找不到关键字符，返回空字符串或提示信息。
    """
    # 查找最后一个斜杠
    last_slash_index = s.rfind('/')
    if last_slash_index == -1:
        file_part = s  # 如果没有斜杠，则整个字符串视为文件名部分
    else:
        file_part = s[last_slash_index + 1:]
    
    # 查找第一个下划线
    underscore_index = file_part.find('_')
    if underscore_index == -1:
        return ""  # 或者 return "字符串中未找到下划线"
    
    after_underscore = file_part[underscore_index + 1:]
    
    # 查找 .csv 后缀
    dotcsv_index = after_underscore.find('.csv')
    if dotcsv_index == -1:
        return after_underscore  # 或者 return "字符串中未找到.csv后缀"
    
    return after_underscore[:dotcsv_index]



In [ ]:


factor_name = extract_content(factor_csv_file_path)
factor_name_pathlib = Path(factor_csv_file_path).stem



# 因子预处理

In [ ]:
your_signal_df = pd.read_csv(factor_csv_file_path)
your_signal_df=your_signal_df.rename(columns={'code': 'instrument','pb':'PB'})
your_signal_df['datetime'] = pd.to_datetime(your_signal_df['datetime'])
your_signal_df[factor_name] = pd.to_numeric(your_signal_df[factor_name], errors='coerce').astype('float32')
multi_df = your_signal_df.set_index(['instrument', 'datetime']).sort_index()

print(len(multi_df))

In [ ]:
from qlib.backtest.signal import create_signal_from
signal_series =multi_df[factor_name].copy()
signal_series = signal_series.dropna()
signal_obj = create_signal_from(signal_series)
print(len(signal_series))
print(signal_obj)



# 策略定义

In [ ]:
# 3. 定义等权策略类（固定调仓周期调仓）
from qlib.contrib.strategy import WeightStrategyBase
from qlib.backtest import backtest, executor

class EqualWeightStrategy(WeightStrategyBase):
    """
    等权策略：选择前topk只股票等权重分配
    """
    def __init__(self, start_percent=0,end_percent=20,topk=20,rebalance_day=20, **kwargs):
        super().__init__(**kwargs)
        self.topk = topk
        self.start_percent=start_percent
        self.end_percent=end_percent
        self.rebalance_day=rebalance_day
        self.trade_day_count=0
        self.pending_sell=[]
        self.coalesced_stocks=[]

    def generate_target_weight_position(self, score, current, trade_start_time, trade_end_time):
        """
        生成目标权重仓位
        
        参数:
            score: pd.Series，索引为股票代码，值为预测分数
            current: 当前持仓对象
            trade_start_time: 交易开始时间
            trade_end_time: 交易结束时间
            
        返回:
            dict，键为股票代码，值为目标权重
        """
        if score is None:
            return 
        
        self.trade_day_count+=1


        # 获取当前持仓
        current_holdings = current.position
        current_position_weights = {}
        for stock_code, stock_info in current_holdings.items():
            if stock_code not in ['cash', 'now_account_value']:
                current_position_weights[stock_code] = stock_info.get('weight', 0)
        
        # print(f"🔍 第{self.trade_day_count}个交易日 {trade_start_time}")
        # print(f"📊 当前持仓股票数: {len(current_position_weights)}")
        # print(f"📋 待卖出列表: {self.pending_sell}")
        # print(f"📈 当前持仓权重: {current_position_weights}")
        # print(f"📈 当前信号值: {score.sort_values(ascending=True).to_dict()}")


        # 1 检查是否合并或者调离指数
        if self.trade_day_count % self.rebalance_day!=0:
    
            # 检查score中是否有-99999和-99998值
            invalid_stocks_99999 = score[score == -99999].index.tolist()
            invalid_stocks_99998 = score[score == -99998].index.tolist()
            
            # 处理-99999的股票（添加到待卖出列表）
            if len(invalid_stocks_99999) > 0:
                # print(f"发现无效信号股票(-99999),出现调离指数: {invalid_stocks_99999}")
                for stock_id in invalid_stocks_99999:
                    if stock_id not in self.pending_sell and stock_id  in current_position_weights:
                        self.pending_sell.append(stock_id)
                        # print(f"已将 {stock_id} 添加到待卖出列表")
            
            # 处理-99998的股票（添加到coalesced_stocks）
            if len(invalid_stocks_99998) > 0:
                # print(f"发现合并股票(-99998): {invalid_stocks_99998}")
                for stock_id in invalid_stocks_99998:
                    
                    if stock_id not in self.coalesced_stocks:
                        self.coalesced_stocks.append(stock_id)
                        # print(f"已将 {stock_id} 添加到coalesced_stocks列表")

                    if stock_id not in self.pending_sell and stock_id  in current_position_weights:
                        self.pending_sell.append(stock_id)
                        # print(f"已将 {stock_id} 添加到待卖出列表")    
                            


        # 2 处理待卖出列表
        if len(self.pending_sell) !=0 and self.trade_day_count % self.rebalance_day!=0:

            # print(f"当前时间为{trade_start_time}，开始处理待卖出列表{self.pending_sell}")
            tradable_pending_sell_stock_id=[]
            tradable_pending_sell_dict={}

            for stock_id in self.pending_sell:
                if self.trade_exchange.is_stock_tradable(
                    stock_id=stock_id,
                    start_time=trade_start_time,
                    end_time=trade_end_time
                ):
                    tradable_pending_sell_stock_id.append(stock_id)
            # print(f"可交易的的待卖出列表中的股票: {tradable_pending_sell_stock_id}")
            for stock_id in tradable_pending_sell_stock_id:
                self.pending_sell.remove(stock_id)
                tradable_pending_sell_dict[stock_id]=0
                current_position_weights[stock_id]=0
            
            # print(f"可卖出的待卖出列表的权重字典{tradable_pending_sell_dict}")
            if len(tradable_pending_sell_dict) > 0:
                return current_position_weights

            else:
                # 修复：如果没有可卖出的股票，返回
                # print(f"没有可卖出的待卖出股票，保持当前持仓不变")
                pass
                

        # 3 调仓日重新选择股票组合
        if self.trade_day_count % self.rebalance_day==0:

            # print(f"🔄 调仓日 - 重新选择股票组合")
            # print(f"交易开始时间{trade_start_time}，交易结束时间{trade_end_time}")
            
            # 现在current_weights就包含了当前所有持仓及其权重
            # print(f"当前持仓权重: {current_position_weights}")
            # print(f'当前持仓股票总数：{len(current_position_weights)}')

            # 选择前 topk 只股票（只从有效信号中选择）
            if len(score) > 0:
                
                valid_score = score[~score.isin([-99999, -99998])] # 默认先过滤特殊值

                # 排除在 coalesced_stocks 中的股票
                if hasattr(self, 'coalesced_stocks') and len(self.coalesced_stocks) > 0:
                    # 在默认值的基础上，再次过滤掉 coalesced_stocks 中的股票
                    valid_score = valid_score[~valid_score.index.isin(self.coalesced_stocks)]
                
                if len(valid_score) > 0:
                    # 选择前topk只股票
                    # actual_topk = min(self.topk, len(valid_score))
                    # sorted_score = valid_score.sort_values(ascending=True)
                    # selected_stocks = sorted_score.head(actual_topk).index
                    # print(f"实际选择股票数量: {actual_topk}")
                    # print(f"当天有效的的股票因子值: {valid_score.sort_values(ascending=True).to_dict()}")
                    # print(f"选择的股票的因子值: {sorted_score.sort_values(ascending=True).to_dict()}")
                    # 选择start_percent和end_percent之间的股票
                    total_stocks = len(valid_score)
                    start_index = int(total_stocks * self.start_percent)
                    end_index = int(total_stocks * self.end_percent)
                    actual_count = end_index - start_index

                    if actual_count > 0:
                        sorted_score = valid_score.sort_values(ascending=True)
                        # 按百分比选择股票
                        selected_stocks = sorted_score.iloc[start_index:end_index].index
                        # print(f"总股票数: {total_stocks}, 选择范围: {start_index}-{end_index}, 实际选择股票数量: {actual_count}")
                        # print(f"当天有效的股票因子值: {valid_score.sort_values(ascending=True).to_dict()}")
                        # print(f"选择的股票因子值: {sorted_score.iloc[start_index:end_index].to_dict()}")
                    else:
                        # print("按百分比计算后没有股票可选")
                        selected_stocks = pd.Index([])
                else:
                    # print("所有信号都是无效值")
                    selected_stocks = pd.Index([])
            else:
                # print("score为空")
                selected_stocks = pd.Index([])

            # print(f"选择的股票数量: {len(selected_stocks)}")
          
            if len(selected_stocks) > 0:
                # print(f"选择的股票: {selected_stocks.tolist()}...")  # 显示前5只
                pass
            else:
                return
            # 为每只股票分配相等的权重，并添加归一化处理
            if len(selected_stocks) > 0:
                # 检查可交易性并筛选可交易股票
                current_tradable_stocks = []
                current_non_tradable_stocks = []

                for stock_id in current_position_weights.keys():
                    if  self.trade_exchange.is_stock_tradable(
                        stock_id=stock_id,
                        start_time=trade_start_time,
                        end_time=trade_end_time
                    ):
                        current_tradable_stocks.append(stock_id)
                    else:
                        current_non_tradable_stocks.append(stock_id)


                # print(f"当前持仓可交易股票数量: {len(current_tradable_stocks)}")
                # print(f"当前持仓可交易股票: {current_tradable_stocks}")

                # print(f"当前持仓不可交易股票数量: {len(current_non_tradable_stocks)}")
                # print(f"当前持仓不可交易股票: {current_non_tradable_stocks}")

                new_group_tradable_stocks = []
                for stock_id in selected_stocks:
                    if self.trade_exchange.is_stock_tradable(
                        stock_id=stock_id,
                        start_time=trade_start_time,
                        end_time=trade_end_time
                    ):
                        new_group_tradable_stocks.append(stock_id)
                        
                # print(f"新分组可交易股票数量: {len(new_group_tradable_stocks)}")
                # print(f"新分组可交易股票: {new_group_tradable_stocks}")

                if len(new_group_tradable_stocks) > 0:
                    # 初始化目标权重字典
                    target_weight_dict = {}

                    # 计算当前持仓无法交易股票的总权重
                    non_tradable_weight_sum = 0
                    for stock_id in current_non_tradable_stocks:
                        non_tradable_weight_sum += current_position_weights[stock_id]
                        #将当前持仓无法交易的股票迁移至目标权重字典(是否无需迁移 因为target_weight_dict中不包含就不会操作)
                        target_weight_dict[stock_id] = current_position_weights[stock_id]
                    # print(f"当前持仓无法交易股票总权重: {non_tradable_weight_sum:.4f}")
                    
                    # 找出不在当前分组但在持仓中且可交易的股票，将其权重设置为0
                    current_position_tradable_stocks = set(current_tradable_stocks)
                    current_position_non_tradable_stocks = set(current_non_tradable_stocks)
                    current_selection_stocks = set(selected_stocks)
                    stocks_to_zero = current_position_tradable_stocks - current_selection_stocks

                    #找出不在当前分组但在持仓中且不可交易的股票，将其加入待卖出列表
                    stock_to_sell=current_position_non_tradable_stocks - current_selection_stocks
                    for stock_id in stock_to_sell:
                        if stock_id not in self.pending_sell:
                            self.pending_sell.append(stock_id)


                    # print(f"当前需权重归零的股票数量: {len(stocks_to_zero)}")

                    # print(f"当前需权重归零的股票: {list(stocks_to_zero)}")
                    for stock_id in stocks_to_zero:
                        target_weight_dict[stock_id] = 0
                    
                    # 计算剩余可交易权重
                    available_weight = 1.0 - non_tradable_weight_sum
                    # print(f"剩余可交易权重: {available_weight:.4f}")
                    
                    # 对可交易股票进行等权分配
                    if available_weight > 0 and len(new_group_tradable_stocks) > 0:
                        equal_weight = available_weight / len(new_group_tradable_stocks)
                        for stock_id in new_group_tradable_stocks:
                            target_weight_dict[stock_id] = equal_weight
                    
                    # print(f"目标权重分配: {target_weight_dict}")
                    # print(f"目标权重字典总权重: {sum(target_weight_dict.values()):.4f}")

                    # print("\n")

                    return target_weight_dict
        else:
            # print(f"⏸️ 非调仓日且无待卖出股票，保持当前持仓")
            return 


In [ ]:
# 4. 构建回测配置
# 策略配置 - 关键：使用正确的信号对象

# 定义目标文件夹路径
folder_path = f'./logs/{factor_name_pathlib}'
# 核心步骤：创建文件夹
os.makedirs(folder_path, exist_ok=True)  

# 执行器配置
executor_config = {
    "time_per_step": "day",
    "generate_portfolio_metrics": True,
}

# 回测参数配置
backtest_config = {
    "start_time": start_time,
    "end_time": end_time,
    "account": 100000000,          # 初始资金 1亿
    "benchmark": extract_index_code_from_filename(factor_csv_file_path),      # 基准指数
    "exchange_kwargs": {
        "limit_threshold": ('$limit_up', '$limit_down'),   # 涨跌停限制
        "deal_price": "close",     # 以收盘价交易
        "open_cost": 0.0005,       # 开仓手续费
        "close_cost": 0.0005,      # 平仓手续费
        "min_cost": 50,             # 最低手续费
    },
}
from datetime import datetime
def process_holdings_data_separated(positions_dict,group_num, output_dir=f'logs/{factor_name_pathlib}'):
    """
    将持仓数据分成两张表：
    1. 每日账户汇总表（现金和账户价值）
    2. 每日股票持仓表（具体持仓信息）
    """
    if not positions_dict:
        print("持仓数据为空")
        return pd.DataFrame(), pd.DataFrame()
    
    # 存储每日账户汇总信息
    daily_account_list = []
    # 存储每日股票持仓信息
    daily_holdings_list = []
    
    # 按日期排序，确保时间顺序
    sorted_dates = sorted(positions_dict.keys())
    
    for i, date_str in enumerate(sorted_dates):
        position_obj = positions_dict[date_str]
        
        try:
            # 安全地获取持仓信息
            if hasattr(position_obj, 'position'):
                holdings_info = position_obj.position
            elif isinstance(position_obj, dict):
                holdings_info = position_obj
            else:
                print(f"未知的持仓对象类型: {type(position_obj)}")
                continue
            
            # 安全地提取现金和总资产，使用get方法避免KeyError
            cash = holdings_info.get('cash', 0)
            account_value = holdings_info.get('now_account_value', 0)
            
            # 如果当前日期没有现金信息，尝试从前一个交易日获取
            if cash == 0 and i > 0:
                prev_date = sorted_dates[i-1]
                prev_position = positions_dict[prev_date]
                if hasattr(prev_position, 'position'):
                    prev_holdings = prev_position.position
                    cash = prev_holdings.get('cash', 0)
                    print(f"从 {prev_date} 获取现金信息: {cash}")
            
            # 创建每日账户汇总记录（表1：账户汇总）
            daily_account_data = {
                'date': pd.to_datetime(date_str),
                'cash': cash,
                'account_value': account_value,
                'stock_count': 0,  # 持股数量
                'total_position_value': 0,  # 总持仓价值
                'cash_ratio': 0,  # 现金比例
                'position_ratio': 0  # 持仓比例
            }
            
            # 遍历该日期持有的每一只股票，统计信息
            stock_count = 0
            total_position_value = 0
            
            for stock_code, stock_info in holdings_info.items():
                if (stock_code not in ['cash', 'now_account_value'] and 
                    isinstance(stock_info, dict) and 
                    stock_info.get('amount', 0) > 0):  # 只处理有持仓的股票
                    
                    # 创建股票持仓记录（表2：股票持仓）
                    stock_data = {
                        'date': pd.to_datetime(date_str),
                        'stock_code': stock_code,
                        'amount': stock_info.get('amount', 0),
                        'price': stock_info.get('price', 0),
                        'weight': stock_info.get('weight', 0),
                        'count_day': stock_info.get('count_day', 0),
                        'position_value': 0
                    }
                    
                    # 计算持仓价值
                    amount = stock_data.get('amount', 0)
                    price = stock_data.get('price', 0)
                    position_value = amount * price
                    stock_data['position_value'] = position_value
                    
                    # 确保所有必要字段都存在
                    for field in ['amount', 'price', 'weight', 'count_day']:
                        if field not in stock_data:
                            stock_data[field] = 0
                    
                    daily_holdings_list.append(stock_data)
                    stock_count += 1
                    total_position_value += position_value
            
            # 更新账户汇总信息
            daily_account_data['stock_count'] = stock_count
            daily_account_data['total_position_value'] = total_position_value
            
            # 计算比例
            if account_value > 0:
                daily_account_data['cash_ratio'] = cash / account_value
                daily_account_data['position_ratio'] = total_position_value / account_value
            
            daily_account_list.append(daily_account_data)
            
            print(f"日期 {date_str}: 现金={cash:.2f}, 账户价值={account_value:.2f}, 持股数={stock_count}, 持仓价值={total_position_value:.2f}")
            
        except Exception as e:
            print(f"处理日期 {date_str} 时出错: {e}")
            continue
    
    # 创建两个DataFrame
    if daily_account_list:
        # 表1：每日账户汇总
        account_df = pd.DataFrame(daily_account_list)
        
        # 确保数值列的数据类型正确
        numeric_columns = ['cash', 'account_value', 'stock_count', 'total_position_value', 'cash_ratio', 'position_ratio']
        for col in numeric_columns:
            if col in account_df.columns:
                account_df[col] = pd.to_numeric(account_df[col], errors='coerce').fillna(0)
        
        # 保存账户汇总表
        account_file = f'{output_dir}/{factor_name_pathlib}_第{group_num}组_每日账户汇总.csv'
        account_df.to_csv(account_file, index=False, encoding='utf-8-sig')
        print(f"每日账户汇总表已保存到: {account_file}")
        
        print("=" * 60)
        print("表1：每日账户汇总")
        print("=" * 60)
        print(f"交易日数: {len(account_df)}")
        print(f"账户价值范围: {account_df['account_value'].min():.2f} - {account_df['account_value'].max():.2f}")
        print(f"现金范围: {account_df['cash'].min():.2f} - {account_df['cash'].max():.2f}")
        print(f"平均持股数: {account_df['stock_count'].mean():.1f}")
        print(f"平均现金比例: {account_df['cash_ratio'].mean():.2%}")
        print(f"平均持仓比例: {account_df['position_ratio'].mean():.2%}")
        
    else:
        account_df = pd.DataFrame()
        print("没有账户汇总数据")
    
    if daily_holdings_list:
        # 表2：每日股票持仓
        holdings_df = pd.DataFrame(daily_holdings_list)
        
        # 确保数值列的数据类型正确
        numeric_columns = ['amount', 'price', 'weight', 'count_day', 'position_value']
        for col in numeric_columns:
            if col in holdings_df.columns:
                holdings_df[col] = pd.to_numeric(holdings_df[col], errors='coerce').fillna(0)
        
        # 保存股票持仓表
        holdings_file = f'{output_dir}/{factor_name_pathlib}_第{group_num}组_每日股票持仓.csv'
        holdings_df.to_csv(holdings_file, index=False, encoding='utf-8-sig')
        print(f"每日股票持仓表已保存到: {holdings_file}")
        
        print("\n" + "=" * 60)
        print("表2：每日股票持仓")
        print("=" * 60)
        print(f"股票持仓记录数: {len(holdings_df)}")
        print(f"涉及股票数: {holdings_df['stock_code'].nunique()}")
        print(f"平均每日持股数: {holdings_df.groupby('date')['stock_code'].count().mean():.1f}")
        
        # 最常持有的股票
        top_stocks = holdings_df['stock_code'].value_counts().head(10)
        print("最常持有的前10只股票:")
        for stock, count in top_stocks.items():
            print(f"  {stock}: {count} 天")
            
    else:
        holdings_df = pd.DataFrame()
        print("没有股票持仓数据")
    
    return account_df, holdings_df
param_dict={
    0:[0,0.2],
    1:[0.2,0.4],
    2:[0.4,0.6],
    3:[0.6,0.8],
    4:[0.8,1]}
for i in range(5):
        print("=== 运行回测 ===")
        try:
            strategy_config = {
                "topk": 20,                    # 选择前20只股票
                "start_percent":param_dict[i][0],
                "end_percent":param_dict[i][1],
                "signal": signal_obj,          # 【重要】使用信号对象而不是原始数据
                "rebalance_day": 20,
                "risk_degree": 1,
            }
            # 实例化策略和执行器
            strategy_obj = EqualWeightStrategy(**strategy_config)
            executor_obj = executor.SimulatorExecutor(**executor_config)

            print(f"策略对象创建完成: {type(strategy_obj)}")
            print(f"执行器对象创建完成: {type(executor_obj)}")
            print(f"信号对象类型: {type(signal_obj)}")

            # 执行回测
            print("开始执行回测...")
            portfolio_metric_dict, indicator_dict = backtest(
                executor=executor_obj,
                strategy=strategy_obj,
                **backtest_config
            )

            print(f"第{i+1}次，回测完成！")
            print(f"可用的回测频率: {list(portfolio_metric_dict.keys())}")      
        except Exception as e:
            print(f"回测出错: {e}")
            import traceback
            print(traceback.format_exc())
            
        report_df, positions_dict_for_analysis = portfolio_metric_dict['1day']
        import datetime
        # 查看绩效报告的列名，确保要用的列都存在
        print(report_df.columns.tolist())

        # 计算并添加一些常用指标，例如累计收益率
        report_df['cumulative_return'] = (1 + report_df['return']).cumprod() - 1
        report_df['bench_cumulative_return'] = (1 + report_df['bench']).cumprod() - 1

        # 选择关心的列输出
        key_metrics_df = report_df[['account', 'return', 'cumulative_return', 'bench', 'bench_cumulative_return', 'turnover', 'total_cost']]

        now_time=datetime.datetime.now()

        print(key_metrics_df.info()) # 查看最后10天的情况
        # 可以将 DataFrame 保存到CSV文件
        key_metrics_df.to_csv(f'./logs/{factor_name_pathlib}/{factor_name_pathlib}_第{i+1}组_回测绩效报告.csv', encoding='utf-8-sig')
        account_df, holdings_df = process_holdings_data_separated(positions_dict=positions_dict_for_analysis,group_num=(i+1))



# 结果输出

In [ ]:

report_df, positions_dict = portfolio_metric_dict['1day']
 

# 可视化部分

In [ ]:
# 可视化分析代码
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from matplotlib.patches import Rectangle
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.offline as pyo
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# 设置字体家族为支持中文的字体
plt.rcParams['font.sans-serif'] = ['SimHei']  # Windows 系统常用黑体
# 解决负号显示问题
plt.rcParams['axes.unicode_minus'] = False


In [ ]:
# 1. 账户价值趋势分析函数 
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime
import numpy as np

def plot_account_value_analysis_improved(report_df,account_df, save_path=f'logs/{factor_name_pathlib}'):
    """
    绘制账户价值趋势分析图表
    包括：净值曲线、回撤分析、滚动收益率和波动率分析
    """
    if account_df.empty:
        print("❌ 账户数据为空，无法绘制图表")
        return
    
    # 确保日期列为datetime类型
    account_df = account_df.copy()
    account_df['date'] = pd.to_datetime(account_df['date'])
    account_df = account_df.sort_values('date')
    
    # 计算净值（相对于初始值）
    initial_value = account_df['account_value'].iloc[0]
    account_df['net_value'] = account_df['account_value'] / initial_value
    
    # 计算回撤
    account_df['cummax'] = account_df['net_value'].cummax()
    account_df['drawdown'] = (account_df['net_value'] - account_df['cummax']) / account_df['cummax']*100
    
    # 计算日收益率
    account_df['daily_return'] = account_df['net_value'].pct_change()
    
    # 计算滚动指标（30天窗口）
    window = 30
    account_df['rolling_return'] = account_df['daily_return'].rolling(window=window).mean() * 252  # 年化
    account_df['rolling_volatility'] = account_df['daily_return'].rolling(window=window).std() * np.sqrt(252)  # 年化
    account_df['rolling_sharpe'] = account_df['rolling_return'] / account_df['rolling_volatility']


    # 计算基准指数回撤
    report_df_copy = report_df.copy()
    report_df_copy['date'] = pd.to_datetime(report_df_copy['datetime'])
    report_df_copy['r_net_value']=report_df_copy['bench_cumulative_return']+1
    report_df_copy['r_cummax']=report_df_copy['r_net_value'].cummax()
    report_df_copy['r_drawdown']=(report_df_copy['r_net_value']-report_df_copy['r_cummax'])/report_df_copy['r_cummax']*100
    
    # 将基准数据合并到account_df中
    account_df = account_df.merge(
        report_df_copy[['date', 'r_drawdown']], 
        on='date', 
        how='left'
    )
    
    


    # 创建子图
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('账户价值趋势分析', fontsize=16, fontweight='bold')
    
    # 1. 净值曲线
    ax1 = axes[0, 0]
    ax1.plot(account_df['date'], account_df['net_value'], linewidth=2, color='#2E86AB', label='净值')
    ax1.plot(report_df_copy['date'],report_df_copy['r_net_value'],linewidth=2,color='#FF6B35',label='基准净值')
    ax1.axhline(y=1, color='red', linestyle='--', alpha=0.7, label='初始价值')
    ax1.set_title('净值曲线', fontsize=14, fontweight='bold')
    ax1.set_ylabel('净值')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # 2. 回撤分析
    ax2 = axes[0, 1]
    ax2.fill_between(account_df['date'], account_df['drawdown'], 0, 
                     color='red', alpha=0.3, label='回测回撤')
    ax2.fill_between(account_df['date'], account_df['r_drawdown'], 0, 
                     color='blue', alpha=0.3, label='基准回撤')    
    ax2.plot(account_df['date'], account_df['drawdown'], color='red', linewidth=1)
    ax2.plot(account_df['date'], account_df['r_drawdown'], color='blue', linewidth=1)
    ax2.set_title('回撤分析', fontsize=14, fontweight='bold')
    ax2.set_ylabel('回撤比例(%)')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # 3. 滚动年化收益率（30天窗口）
    ax3 = axes[1, 0]
    ax3.plot(account_df['date'], account_df['rolling_return'] * 100, 
             linewidth=2, color='#28A745', label=f'{window}日滚动收益率')
    ax3.axhline(y=0, color='black', linestyle='-', alpha=0.5)
    ax3.set_title(f'滚动年化收益率（{window}日窗口）', fontsize=14, fontweight='bold')
    ax3.set_ylabel('年化收益率 (%)')
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    
    # 4. 换手率
    # 处理换手率数据：去除为0的值和第一次的非0值
    turnover_data = report_df_copy['turnover'].copy()
    # 找到第一个非0值的索引
    first_nonzero_idx = turnover_data[turnover_data > 0].index
    if len(first_nonzero_idx) > 0:
        first_nonzero_idx = first_nonzero_idx[0]
        # 将第一个非0值也设为0（去除）
        turnover_data.iloc[first_nonzero_idx] = 0

    # 去除所有为0的值
    processed_turnover = turnover_data[turnover_data > 0]

    ax4 = axes[1, 1]
    if len(processed_turnover) > 0:
        # 获取对应的日期数据
        turnover_dates = report_df_copy.loc[processed_turnover.index, 'date']
        ax4.plot(turnover_dates, processed_turnover * 100, 
                linewidth=2, color='orange', label='换手率')
        ax4.axhline(y=processed_turnover.mean() * 100, color='red', 
                    linestyle='--', alpha=0.7, label=f'平均换手率: {processed_turnover.mean() * 100:.2f}%')
    else:
        # 如果没有有效数据，显示提示信息
        ax4.text(0.5, 0.5, '无有效换手率数据', transform=ax4.transAxes, 
                ha='center', va='center', fontsize=12)

    ax4.set_title('换手率分析', fontsize=14, fontweight='bold')
    ax4.set_ylabel('换手率 (%)')
    ax4.legend()
    ax4.grid(True, alpha=0.3)

    
    # 设置x轴日期格式
    for ax in axes.flat:
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
        ax.xaxis.set_major_locator(mdates.YearLocator())
        plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, fontsize=8)
    
    plt.tight_layout()
    
    # 保存图表
    timestamp = datetime.now().strftime('%Y%m%d_%H%M')
    filename = f'{save_path}/{factor_name_pathlib}_账户价值分析.png'
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    print(f"📊 改进版账户价值分析图表已保存: {filename}")
    
    plt.show()
    
    # 打印关键统计信息
    print("\n" + "="*60)
    print("📈 关键账户价值指标")
    print("="*60)
    print(f"初始账户价值: {initial_value:,.2f}")
    print(f"最终账户价值: {account_df['account_value'].iloc[-1]:,.2f}")
    print(f"总收益率: {(account_df['net_value'].iloc[-1] - 1) * 100:.2f}%")
    print(f"最大回撤: {account_df['drawdown'].min() * 100:.2f}%")
    print(f"平均现金比例: {account_df['cash_ratio'].mean() * 100:.2f}%")
    print(f"平均持仓比例: {account_df['position_ratio'].mean() * 100:.2f}%")
    
    # 额外风险指标
    print(f"\n📊 风险指标:")
    print(f"年化收益率: {account_df['daily_return'].mean() * 252 * 100:.2f}%")
    print(f"年化波动率: {account_df['daily_return'].std() * np.sqrt(252) * 100:.2f}%")
    print(f"夏普比率: {(account_df['daily_return'].mean() * 252) / (account_df['daily_return'].std() * np.sqrt(252)):.2f}")
    print(f"最大日亏损: {account_df['daily_return'].min() * 100:.2f}%")
    print(f"最大日收益: {account_df['daily_return'].max() * 100:.2f}%")
    
    # 滚动指标汇总
    print(f"\n📈 滚动指标（30日窗口）:")
    print(f"平均滚动收益率: {account_df['rolling_return'].mean() * 100:.2f}%")
    print(f"平均滚动波动率: {account_df['rolling_volatility'].mean() * 100:.2f}%")
    print(f"平均滚动夏普比率: {account_df['rolling_sharpe'].mean():.2f}")
    print(f"最大滚动夏普比率: {account_df['rolling_sharpe'].max():.2f}")
    print(f"最小滚动夏普比率: {account_df['rolling_sharpe'].min():.2f}")
    
    return fig




In [ ]:
# 2. 风险分析函数
def plot_risk_analysis_fixed(account_df, save_path=f'logs/{factor_name_pathlib}'):
    """
    绘制风险分析图表（中文标签）
    包括：日收益率分布、滚动波动率、夏普比率、VaR分析
    """
    if account_df.empty:
        print("❌ 账户数据为空，无法绘制图表")
        return
    
    # 确保日期列为datetime类型并排序
    account_df = account_df.copy()
    account_df['date'] = pd.to_datetime(account_df['date'])
    account_df = account_df.sort_values('date')
    
    # 计算日收益率
    account_df['daily_return'] = account_df['account_value'].pct_change()
    account_df['daily_return_pct'] = account_df['daily_return'] * 100
    
    # 计算滚动波动率（30天窗口）
    account_df['rolling_volatility'] = account_df['daily_return'].rolling(window=30).std() * np.sqrt(252) * 100
    
    # 计算滚动夏普比率（30天窗口，假设无风险利率为3%）
    risk_free_rate = 0.03
    account_df['rolling_sharpe'] = (account_df['daily_return'].rolling(window=30).mean() * 252 - risk_free_rate) / (account_df['daily_return'].rolling(window=30).std() * np.sqrt(252))
    
    # 创建子图
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('风险分析', fontsize=16, fontweight='bold')
    
    # 1. 日收益率分布
    ax1 = axes[0, 0]
    returns_data = account_df['daily_return_pct'].dropna()
    ax1.hist(returns_data, bins=50, alpha=0.7, color='#2E86AB', edgecolor='black', density=True)
    ax1.axvline(returns_data.mean(), color='red', linestyle='--', linewidth=2, label=f'均值: {returns_data.mean():.2f}%')
    ax1.axvline(returns_data.median(), color='orange', linestyle='--', linewidth=2, label=f'中位数: {returns_data.median():.2f}%')
    ax1.set_title('日收益率分布', fontsize=14, fontweight='bold')
    ax1.set_xlabel('日收益率 (%)')
    ax1.set_ylabel('密度')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # 2. 滚动波动率
    ax2 = axes[0, 1]
    ax2.plot(account_df['date'], account_df['rolling_volatility'], 
             linewidth=2, color='#A23B72', label='30日滚动波动率')
    ax2.axhline(y=account_df['rolling_volatility'].mean(), color='red', 
                linestyle='--', alpha=0.7, label=f'平均波动率: {account_df["rolling_volatility"].mean():.2f}%')
    ax2.set_title('滚动波动率', fontsize=14, fontweight='bold')
    ax2.set_ylabel('年化波动率 (%)')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # 3. 滚动夏普比率
    ax3 = axes[1, 0]
    ax3.plot(account_df['date'], account_df['rolling_sharpe'], 
             linewidth=2, color='#F18F01', label='30日滚动夏普比率')
    ax3.axhline(y=account_df['rolling_sharpe'].mean(), color='red', 
                linestyle='--', alpha=0.7, label=f'滚动平均夏普比率: {account_df["rolling_sharpe"].mean():.2f}')
    ax3.axhline(y=1, color='green', linestyle='--', alpha=0.7, label='夏普比率=1')
    ax3.set_title('滚动夏普比率', fontsize=14, fontweight='bold')
    ax3.set_ylabel('夏普比率')
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    
    # 4. VaR分析（风险价值）
    ax4 = axes[1, 1]
    # 计算不同置信水平的VaR
    confidence_levels = [0.95, 0.99]
    var_values = []
    var_labels = []
    
    for conf in confidence_levels:
        var_value = np.percentile(returns_data, (1 - conf) * 100)
        var_values.append(var_value)
        var_labels.append(f'{conf*100}% VaR: {var_value:.2f}%')
    
    # 绘制VaR线
    ax4.hist(returns_data, bins=50, alpha=0.7, color='#2E86AB', edgecolor='black', density=True)
    for i, (var_val, var_label) in enumerate(zip(var_values, var_labels)):
        ax4.axvline(var_val, color=['red', 'darkred'][i], linestyle='--', 
                   linewidth=2, label=var_label)
    
    ax4.set_title('VaR分析', fontsize=14, fontweight='bold')
    ax4.set_xlabel('日收益率 (%)')
    ax4.set_ylabel('密度')
    ax4.legend()
    ax4.grid(True, alpha=0.3)
    
    # 设置x轴日期格式
    for ax in [axes[0, 1], axes[1, 0]]:
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
        ax.xaxis.set_major_locator(mdates.YearLocator())
        plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, fontsize=8)
    
    plt.tight_layout()
    
    # 保存图表
    timestamp = datetime.now().strftime('%Y%m%d_%H%M')
    filename = f'{save_path}/{factor_name_pathlib}_风险分析.png'
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    print(f"📊 风险分析图表已保存: {filename}")
    
    plt.show()
    
    # 打印关键风险指标
    print("\n" + "="*60)
    print("⚠️ 关键风险分析指标")
    print("="*60)
    print(f"平均日收益率: {returns_data.mean():.4f}%")
    print(f"日收益率标准差: {returns_data.std():.4f}%")
    print(f"年化波动率: {returns_data.std() * np.sqrt(252):.2f}%")
    print(f"夏普比率: {(returns_data.mean() * 252 - risk_free_rate) / (returns_data.std() * np.sqrt(252)):.2f}")
    print(f"最大日亏损: {returns_data.min():.2f}%")
    print(f"最大日收益: {returns_data.max():.2f}%")
    print(f"95% VaR: {var_values[0]:.2f}%")
    print(f"99% VaR: {var_values[1]:.2f}%")
    
    # 计算偏度和峰度
    from scipy import stats
    skewness = stats.skew(returns_data)
    kurtosis = stats.kurtosis(returns_data)
    print(f"偏度: {skewness:.2f}")  
    print(f"峰度: {kurtosis:.2f}")
    
    return fig




In [ ]:
# 3. 收益分析函数
def plot_returns_analysis_english(report_df_group1, account_df_group1,report_df_group5,account_df_group5,save_path=f'logs/{factor_name_pathlib}'):
    """
    绘制收益分析图表（中文标签）
    包括：累计收益率曲线、年化收益率、月度收益率热力图、与基准对比
    """
    if account_df_group1.empty:
        print("❌ 账户数据为空，无法绘制图表")
        return
    
    # 确保日期列为datetime类型并排序
    account_df_group1['date'] = pd.to_datetime(account_df_group1['date'])
    account_df_group1 = account_df_group1.sort_values('date')
    
    # 计算日收益率和累计收益率
    account_df_group1['daily_return'] = account_df_group1['account_value'].pct_change()
    account_df_group1['cumulative_return'] = (1 + account_df_group1['daily_return']).cumprod() - 1
    account_df_group1['cumulative_return_pct'] = account_df_group1['cumulative_return'] * 100

    # 确保日期列为datetime类型并排序
    account_df_group5['date'] = pd.to_datetime(account_df_group5['date'])
    account_df_group5 = account_df_group5.sort_values('date')
    
    # 计算日收益率和累计收益率
    account_df_group5['daily_return'] = account_df_group5['account_value'].pct_change()
    account_df_group5['cumulative_return'] = (1 + account_df_group5['daily_return']).cumprod() - 1
    account_df_group5['cumulative_return_pct'] = account_df_group5['cumulative_return'] * 100


    # 准备基准数据：将report_df的datetime列转换为日期格式进行匹配
    report_df_group1_copy = report_df_group1.copy()
    report_df_group1_copy['date'] = pd.to_datetime(report_df_group1['datetime'])
    
    # 将基准数据合并到account_df中
    account_df_group1 = account_df_group1.merge(
        report_df_group1_copy[['date', 'bench', 'bench_cumulative_return']], 
        on='date', 
        how='left'
    )
    
    # 计算基准的百分比形式
    account_df_group1['bench_cumulative_return_pct'] = account_df_group1['bench_cumulative_return'] * 100
    account_df_group1['bench_daily_return_pct'] = account_df_group1['bench'] * 100
    
    # 计算年化收益率
    total_days = (account_df_group1['date'].iloc[-1] - account_df_group1['date'].iloc[0]).days
    years = total_days / 365.25
    annualized_return = (account_df_group1['account_value'].iloc[-1] / account_df_group1['account_value'].iloc[0]) ** (1/years) - 1
    
    # 创建子图
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('收益分析', fontsize=16, fontweight='bold')
    
    # 1. 累计收益率曲线
    ax1 = axes[0, 0]
    ax1.plot(account_df_group1['date'], account_df_group1['cumulative_return_pct'], 
             linewidth=2, color='#2E86AB', label='group_1策略累计收益率')
    ax1.plot(account_df_group5['date'], account_df_group5['cumulative_return_pct'], 
            linewidth=2, color='#FFF44F', label='group5_策略累计收益率')
    # 添加基准累计收益率曲线
    ax1.plot(account_df_group1['date'], account_df_group1['bench_cumulative_return_pct'], 
             linewidth=2, color='#FF6B35', label='基准累计收益率')
    ax1.axhline(y=0, color='red', linestyle='--', alpha=0.7, label='零收益线')
    ax1.set_title('累计收益率曲线 vs 基准', fontsize=14, fontweight='bold')
    ax1.set_ylabel('累计收益率 (%)')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # 2. 日收益率时间序列
    ax2 = axes[0, 1]
    ax2.plot(account_df_group1['date'], account_df_group1['daily_return'] * 100, 
             linewidth=1, color='#A23B72', alpha=0.7, label='策略日收益率')
    # # 添加基准日收益率曲线
    # ax2.plot(account_df_group1['date'], account_df_group1['bench_daily_return_pct'], 
    #          linewidth=1, color='#FF6B35', alpha=0.1, label='基准日收益率')
    ax2.axhline(y=0, color='red', linestyle='--', alpha=0.7)
    ax2.set_title('日收益率时间序列', fontsize=14, fontweight='bold')
    ax2.set_ylabel('日收益率 (%)')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # 3. 月度收益率热力图
    ax3 = axes[1, 0]
    # 创建月度收益率数据
    account_df_group1['year'] = account_df_group1['date'].dt.year
    account_df_group1['month'] = account_df_group1['date'].dt.month
    
    monthly_returns = account_df_group1.groupby(['year', 'month'])['daily_return'].apply(lambda x: (1 + x).prod() - 1) * 100
    monthly_returns = monthly_returns.reset_index()
    monthly_returns = monthly_returns.pivot(index='year', columns='month', values='daily_return')
    
    if not monthly_returns.empty:
        sns.heatmap(monthly_returns, annot=True, fmt='.1f', cmap='RdYlGn', 
                   center=0, ax=ax3, cbar_kws={'label': '月度收益率 (%)'})
        ax3.set_title('月度收益率热力图', fontsize=14, fontweight='bold')
        ax3.set_xlabel('月份')
        ax3.set_ylabel('年份')
    else:
        ax3.text(0.5, 0.5, '无月度数据', ha='center', va='center', transform=ax3.transAxes)
        ax3.set_title('月度收益率热力图', fontsize=14, fontweight='bold')
    
    # 4. 年度收益率分布箱线图
    ax4 = axes[1, 1]
    yearly_returns = []
    yearly_labels = []
    
    for year in sorted(account_df_group1['year'].unique()):
        year_data = account_df_group1[account_df_group1['year'] == year]['daily_return'] * 100
        if not year_data.empty:
            yearly_returns.append(year_data.dropna())
            yearly_labels.append(str(year))
    
    if yearly_returns:
        ax4.boxplot(yearly_returns, labels=yearly_labels)
        ax4.set_title('年度收益率分布箱线图', fontsize=14, fontweight='bold')
        ax4.set_xlabel('年份')
        ax4.set_ylabel('日收益率 (%)')
        ax4.grid(True, alpha=0.3)
    else:
        ax4.text(0.5, 0.5, '无年度数据', ha='center', va='center', transform=ax4.transAxes)
        ax4.set_title('年度收益率分布箱线图', fontsize=14, fontweight='bold')
    
    # 设置x轴日期格式
    for ax in [axes[0, 0], axes[0, 1]]:
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
        ax.xaxis.set_major_locator(mdates.YearLocator())
        plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, fontsize=8)
    
    plt.tight_layout()
    
    # 保存图表
    from datetime import datetime
    timestamp = datetime.now().strftime('%Y%m%d_%H%M')
    filename = f'{save_path}/{factor_name_pathlib}_收益分析.png'
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    print(f"📊 收益分析图表已保存: {filename}")
    
    plt.show()
    
    # 打印关键收益指标
    # print("\n" + "="*60)
    # print("💰 关键收益分析指标")
    # print("="*60)
    # print(f"总收益率: {account_df_group1['cumulative_return_pct'].iloc[-1]:.2f}%")
    # print(f"年化收益率: {annualized_return * 100:.2f}%")
    # print(f"最大日收益: {account_df_group1['daily_return'].max() * 100:.2f}%")
    # print(f"最大日亏损: {account_df_group1['daily_return'].min() * 100:.2f}%")
    # print(f"平均日收益率: {account_df_group1['daily_return'].mean() * 100:.4f}%")
    # print(f"收益率标准差: {account_df_group1['daily_return'].std() * 100:.4f}%")
    
    # 计算胜率
    positive_days = (account_df_group1['daily_return'] > 0).sum()
    total_days = len(account_df_group1['daily_return'].dropna())
    win_rate = positive_days / total_days if total_days > 0 else 0
    # print(f"胜率: {win_rate * 100:.2f}%")
    
    # 计算最大连续亏损天数
    daily_returns = account_df_group1['daily_return'].dropna()
    consecutive_losses = 0
    max_consecutive_losses = 0
    for ret in daily_returns:
        if ret < 0:
            consecutive_losses += 1
            max_consecutive_losses = max(max_consecutive_losses, consecutive_losses)
        else:
            consecutive_losses = 0
    
    # print(f"最大连续亏损天数: {max_consecutive_losses}")
    
    return fig



In [ ]:
# 多分组累计收益可视化函数
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd
import numpy as np
from datetime import datetime
import seaborn as sns

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

def plot_multi_group_cumulative_returns(df, save_path=f'logs/{factor_name_pathlib}', figsize=(15, 10)):
    """
    绘制多分组累计收益对比图表
    
    参数:
    df: 包含cumulative_return_1到cumulative_return_5字段的DataFrame
    save_path: 保存路径，如果为None则不保存
    figsize: 图表尺寸
    """
    # 检查数据
    if df.empty:
        print("❌ 数据为空，无法绘制图表")
        return
    
    # 确保索引为日期格式
    df = df.copy()
    if not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index)
    
    # 提取累计收益列
    cumulative_cols = [col for col in df.columns if col.startswith('cumulative_return_')]
    if not cumulative_cols:
        print("❌ 未找到cumulative_return_字段")
        return
    
    # 按组号排序
    cumulative_cols.sort(key=lambda x: int(x.split('_')[-1]))
    
    print(f"📊 找到 {len(cumulative_cols)} 个分组: {cumulative_cols}")
    


    # 创建图表
   
    colors = 	['#E74C3C', '#3498DB', '#2ECC71', '#F39C12', '#9B59B6']
    fig, axes = plt.subplots(2, 2, figsize=figsize)
    fig.suptitle('多分组累计收益分析', fontsize=16, fontweight='bold')
    
    # 1. 所有分组累计收益对比
    ax1 = axes[0, 0]
    
    for i, col in enumerate(cumulative_cols):
        group_num = col.split('_')[-1]
        ax1.plot(df.index, df[col] * 100, 
                linewidth=2, color=colors[i], 
                label=f'第{group_num}组', alpha=1)
    
    ax1.axhline(y=0, color='black', linestyle='--', alpha=0.5, label='零收益线')
    ax1.set_title('各分组累计收益对比', fontsize=14, fontweight='bold')
    ax1.set_ylabel('累计收益率 (%)')
    ax1.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    ax1.grid(True, alpha=0.3)
    
    # 设置x轴日期格式
    ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    ax1.xaxis.set_major_locator(mdates.YearLocator())
    plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45)
    
    # 2. 五组净值曲线
    ax2 = axes[0, 1]

    for i, col in enumerate(cumulative_cols):
        group_num = col.split('_')[-1]
        # 计算净值（相对于初始值）
        net_value = (1 + df[col]) / (1 + df[col].iloc[0])
        ax2.plot(df.index, net_value, 
                linewidth=2, color=colors[i], 
                label=f'第{group_num}组净值', alpha=1)

    ax2.axhline(y=1, color='black', linestyle='--', alpha=0.5, label='初始净值')
    ax2.set_title('各分组净值曲线', fontsize=14, fontweight='bold')
    ax2.set_ylabel('净值')
    ax2.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    ax2.grid(True, alpha=0.3)

    # 设置x轴日期格式
    ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    ax2.xaxis.set_major_locator(mdates.YearLocator())
    plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45)

    # 3. 第一组和第五组净值曲线及比值
    ax3 = axes[1, 0]

    # 检查是否有第1组和第5组的数据
    group1_col = 'cumulative_return_1'
    group5_col = 'cumulative_return_5'

    if group1_col in df.columns and group5_col in df.columns:
        # 计算净值
        net_value_1 = (1 + df[group1_col]) / (1 + df[group1_col].iloc[0])
        net_value_5 = (1 + df[group5_col]) / (1 + df[group5_col].iloc[0])
        
        # 绘制净值曲线
        # ax3.plot(df.index, net_value_1, 
        #         linewidth=2, color=colors[0], 
        #         label='第1组净值', alpha=1)
        # ax3.plot(df.index, net_value_5, 
        #         linewidth=2, color=colors[4] if len(colors) > 4 else colors[-1], 
        #         label='第5组净值', alpha=1)
        
        # 计算比值（第1组/第5组）
        ratio = net_value_1 / net_value_5
        ax3.plot(df.index, ratio, 
                linewidth=2, color='green', 
                label='第1组/第5组比值', alpha=0.8, linestyle='-')
        
        ax3.axhline(y=1, color='black', linestyle='--', alpha=0.5, label='基准线')
        ax3.set_title('第1组vs第5组净值对比及比值', fontsize=14, fontweight='bold')
        ax3.set_ylabel('净值/比值')
        ax3.legend()
        ax3.grid(True, alpha=0.3)
        
        # 设置x轴日期格式
        ax3.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
        ax3.xaxis.set_major_locator(mdates.YearLocator())
        plt.setp(ax3.xaxis.get_majorticklabels(), rotation=45)
        
        # 打印比值统计信息
        print(f"\n📊 第1组/第5组比值统计:")
        print(f"平均比值: {ratio.mean():.4f}")
        print(f"最大比值: {ratio.max():.4f}")
        print(f"最小比值: {ratio.min():.4f}")
        print(f"最终比值: {ratio.iloc[-1]:.4f}")
        
    else:
        ax3.text(0.5, 0.5, '缺少第1组或第5组数据', 
                ha='center', va='center', transform=ax3.transAxes, fontsize=12)
        ax3.set_title('第1组vs第5组净值比值', fontsize=14, fontweight='bold')

  # 4. 收益统计对比表
    ax4 = axes[1, 1]
    ax4.axis('off')

    # 计算统计指标
    stats_data = []
    for col in cumulative_cols:
        group_num = col.split('_')[-1]
        data = df[col] * 100
        
        # 计算年化收益率
        # 假设数据是按日计算的，年化收益率 = (1 + 最终累计收益率)^(252/总天数) - 1
        total_days = len(data)
        final_return = df[col].iloc[-1]  # 最终累计收益率（小数形式）
        annualized_return = (1 + final_return) ** (252 / total_days) - 1
        
        stats_data.append([
            f'第{group_num}组',
            f'{data.iloc[-1]:.2f}%',  # 最终收益
            f'{annualized_return * 100:.2f}%',  # 年化收益
            f'{data.mean():.2f}%',    # 平均收益
            f'{data.std():.2f}%',     # 标准差
            f'{data.min():.2f}%',     # 最小收益
            f'{data.max():.2f}%'      # 最大收益
        ])

    # 创建表格
    table_data = [['分组', '最终收益', '年化收益', '平均收益', '标准差', '最小收益', '最大收益']] + stats_data
    table = ax4.table(cellText=table_data, cellLoc='center', loc='center',
                    colWidths=[0.12, 0.12, 0.12, 0.12, 0.12, 0.12, 0.12])
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1, 2)

    # 设置表头样式
    for i in range(len(table_data[0])):
        table[(0, i)].set_facecolor('#4CAF50')
        table[(0, i)].set_text_props(weight='bold', color='white')

    # 设置数据行颜色
    for i in range(1, len(table_data)):
        for j in range(len(table_data[0])):
            if i % 2 == 0:
                table[(i, j)].set_facecolor('#f0f0f0')

    ax4.set_title('收益统计对比', fontsize=14, fontweight='bold', pad=20)
    
    plt.tight_layout()
    
    # 保存图表
    if save_path:
        plt.savefig(f'{save_path}/{factor_name_pathlib}_分组回测分析.png', dpi=900, bbox_inches='tight')
        print(f"📈 图表已保存至: {save_path}")
    
    plt.show()
    
    # 打印简要统计
    print("\n📊 各分组收益统计:")
    print("-" * 60)
    for i, col in enumerate(cumulative_cols):
        group_num = col.split('_')[-1]
        final_return = df[col].iloc[-1] * 100
        mean_return = df[col].mean() * 100
        std_return = df[col].std() * 100
        print(f"第{group_num}组: 最终收益 {final_return:.2f}%, 平均收益 {mean_return:.2f}%, 标准差 {std_return:.2f}%")

print("✅ 多分组累计收益可视化函数已定义")


In [ ]:
# 正确初始化一个空的DataFrame，作为合并的起点
combined_df = pd.DataFrame()

for i in range(1, 6):
    # 构建文件路径并读取CSV
    file_path = f'./logs/{factor_name_pathlib}/{factor_name_pathlib}_第{i}组_回测绩效报告.csv'
    df = pd.read_csv(file_path, index_col='datetime')  # 假设'datetime'是公共的日期时间列
    
    # 为当前DataFrame的每一列（除了可能用作索引的列）添加组别后缀
    df_suffixed = df.add_suffix(f'_{i}')
    
    # 将处理后的DataFrame与主DataFrame进行横向合并
    combined_df = pd.concat([combined_df, df_suffixed], axis=1)

# 重置索引，使datetime重新成为一列（如果需要的话）
# combined_df = combined_df.reset_index()
print(combined_df.info())
print(combined_df.iloc[-1:])

In [ ]:
plot_multi_group_cumulative_returns(combined_df)

In [ ]:
from qlib.backtest import report


account_df_group1=pd.read_csv(f'./logs/{factor_name_pathlib}/{factor_name_pathlib}_第1组_每日账户汇总.csv')
report_df_group1=pd.read_csv(f'./logs/{factor_name_pathlib}/{factor_name_pathlib}_第1组_回测绩效报告.csv')

account_df_group5=pd.read_csv(f'./logs/{factor_name_pathlib}/{factor_name_pathlib}_第5组_每日账户汇总.csv')
report_df_group5=pd.read_csv(f'./logs/{factor_name_pathlib}/{factor_name_pathlib}_第5组_回测绩效报告.csv')
# 调用函数绘制改进版账户价值分析图表
if not account_df_group1.empty and not report_df_group1.empty:
    plot_account_value_analysis_improved(report_df_group1,account_df_group1)
    plot_returns_analysis_english(report_df_group1,account_df_group1,report_df_group5,account_df_group5)
    plot_risk_analysis_fixed(account_df_group1)
else:
    print("❌ 账户数据为空，请先运行持仓数据处理代码")

# 计算单调性

In [ ]:
from scipy.stats import spearmanr
# 更灵活的Group IC计算函数
def calculate_group_ic_flexible(df, start_date=None, end_date=None, trading_days_per_year=252):
    """
    计算Group IC指标（灵活版本）
    
    Args:
        df: DataFrame，包含5个组的账户价值数据
        start_date: 开始日期（可选，用于计算实际年数）
        end_date: 结束日期（可选，用于计算实际年数）
        trading_days_per_year: 每年交易日数，默认252
    
    Returns:
        DataFrame: 包含group_ic列的新DataFrame
    """
    result_df = df.copy()
    
    # 计算实际年数
    if start_date is not None and end_date is not None:
        # 如果有具体的开始和结束日期
        if isinstance(start_date, str):
            start_date = pd.to_datetime(start_date)
        if isinstance(end_date, str):
            end_date = pd.to_datetime(end_date)
        
        total_days = (end_date - start_date).days
        years = total_days /252  
        print(f"📅 基于日期计算年数: {years:.2f}年")
    else:
        # 基于数据长度计算
        total_trading_days = len(df)
        years = total_trading_days / trading_days_per_year
        print(f"📊 基于交易日数计算年数: {years:.2f}年")
    
    # 为每个组计算年化收益率
    annualized_returns = []
    group_numbers = []
    
    for i in range(1, 6):  # 组1到组5
        cumulative_return_col = f'cumulative_return_{i}'
        if cumulative_return_col in df.columns:
            # 获取最终累计收益率
            final_cumulative_return = df[cumulative_return_col].iloc[-1]
            
            # 计算年化收益率：(1+总收益率)^(1/年数) 
            annualized_return = (1+final_cumulative_return) ** (1/years) - 1
            annualized_returns.append(annualized_return)
            group_numbers.append(i)
            
            print(f"组{i}: 累计收益率={final_cumulative_return*100:.2f}%, 年化收益率={annualized_return*100:.2f}%")
    
    # 计算Rank IC
    if len(annualized_returns) >= 2:
        # 使用Spearman秩相关系数计算rank IC
        rank_ic, p_value = spearmanr(group_numbers, annualized_returns)
        print(f"Rank IC: {rank_ic:.4f}")
        print(f"P值: {p_value:.4f}")
           
        return rank_ic



group_rank_ic = calculate_group_ic_flexible(combined_df, start_date = start_time , end_date=end_time)


# 输出PDF报告

In [ ]:
# 使用ReportLab生成包含图片和文字说明的PDF报告
from reportlab.lib.pagesizes import A4
from reportlab.platypus import SimpleDocTemplate, Paragraph, Image, Spacer
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib import colors
from reportlab.lib.units import cm
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont
import os
# 尝试注册常见的中文字体
pdfmetrics.registerFont(TTFont('SimHei', 'C:/Windows/Fonts/simhei.ttf'))
font_name = 'SimHei'


# 定义图片路径和对应的说明文字
image_descriptions = [
    {
        'path': f'./logs/{factor_name_pathlib}/{factor_name_pathlib}_账户价值分析.png',
        'title': '账户价值分析',
        'description': '本图表展示了投资组合的账户价值变化趋势，包括净值曲线、回撤分析和滚动收益率。通过此图可以直观了解投资策略的长期表现和风险控制效果。'
    },
    {
        'path': f'./logs/{factor_name_pathlib}/{factor_name_pathlib}_风险分析.png',
        'title': '风险分析',
        'description': '风险分析图表包含了日收益率分布、滚动波动率、夏普比率和VaR分析。这些指标帮助我们评估投资策略的风险水平和风险调整后的收益表现。'
    },
    {
        'path': f'./logs/{factor_name_pathlib}/{factor_name_pathlib}_收益分析.png',
        'title': '收益分析',
        'description': '收益分析展示了累计收益率曲线、年化收益率、月度收益率热力图以及与基准的对比。通过多维度分析，全面评估投资策略的收益特征。'
    },
    {
        'path': f'./logs/{factor_name_pathlib}/{factor_name_pathlib}_分组回测分析.png',
        'title': '分组回测分析',
        'description': '分组回测分析对比了不同分组的累计收益表现，展示了策略在不同市场环境下的适应性和各组别之间的相对表现差异。'
    }
]

# 创建PDF文档
pdf_output_path = f"./logs/{factor_name_pathlib}/{factor_name_pathlib}_分析报告.pdf"
doc = SimpleDocTemplate(pdf_output_path, pagesize=A4, 
                       rightMargin=2*cm, leftMargin=2*cm, 
                       topMargin=2*cm, bottomMargin=2*cm)
story = []

# 准备样式
styles = getSampleStyleSheet()
title_style = styles['Heading1'].clone('title_style')
title_style.fontName = font_name
title_style.alignment = 1  # 居中
title_style.fontSize = 16
title_style.spaceAfter = 20

subtitle_style = styles['Heading2'].clone('subtitle_style')
subtitle_style.fontName = font_name
subtitle_style.fontSize = 14
subtitle_style.spaceAfter = 10
subtitle_style.spaceBefore = 15

body_style = styles['Normal'].clone('body_style')
body_style.fontName = font_name
body_style.fontSize = 10
body_style.leading = 15
body_style.spaceAfter = 10

# 添加报告标题
main_title = Paragraph(f"{factor_name_pathlib} 因子分析报告", title_style)
story.append(main_title)
story.append(Spacer(1, 20))

# 添加报告说明
# intro_text = Paragraph("本报告基于量化投资策略的回测结果，通过多维度分析展示策略的收益表现、风险特征和投资价值。报告包含账户价值分析、风险分析、收益分析和分组回测分析四个核心部分。", body_style)
# story.append(intro_text)
# story.append(Spacer(1, 20))

# 遍历图片和说明，添加到PDF中
for i, item in enumerate(image_descriptions, 1):
    # 检查图片文件是否存在
    if os.path.exists(item['path']):
        # 添加图片标题
        # img_title = Paragraph(f"{i}. {item['title']}", subtitle_style)
        # story.append(img_title)
        
        # 添加图片说明
        # img_desc = Paragraph(item['description'], body_style)
        # story.append(img_desc)
        # story.append(Spacer(1, 10))
        
        # 添加图片
        try:
            img = Image(item['path'], width=16*cm, height=11*cm)
            story.append(img)
            story.append(Spacer(1, 20))
            print(f"✅ 已添加图片: {item['title']}")
        except Exception as e:
            error_text = Paragraph(f"图片加载失败: {str(e)}", body_style)
            story.append(error_text)
            print(f"❌ 图片加载失败: {item['path']} - {str(e)}")
    else:
        # 如果图片不存在，添加提示信息
        missing_text = Paragraph(f"{i}. {item['title']} - 图片文件不存在", body_style)
        story.append(missing_text)
        print(f"⚠️ 图片文件不存在: {item['path']}")
# 添加group_rankIC
group_rank_ic = calculate_group_ic_flexible(combined_df, start_date = start_time , end_date=end_time)

main_title = Paragraph(f"group_rank_ic:{group_rank_ic}", title_style)
story.append(main_title)
story.append(Spacer(1, 20))

# 添加结论
# conclusion_title = Paragraph("分析结论", subtitle_style)
# story.append(conclusion_title)
# conclusion_text = Paragraph("通过以上四个维度的分析，我们可以全面评估投资策略的表现。建议结合具体的风险偏好和投资目标，对策略参数进行进一步优化调整。", body_style)
# story.append(conclusion_text)

# 生成PDF
try:
    doc.build(story)
    print(f"📊 PDF报告已成功生成：{pdf_output_path}")
except Exception as e:
    print(f"❌ PDF生成失败：{str(e)}")